# 🚀 Carnet de mission Python n°2 — Blocs 4, 5 et 6

La suite du carnet n°1 ! **Ce carnet marche sans internet.**

| Bloc | Sujet | Contenu |
|---|---|---|
| 4 | 🧩 Les fonctions | cours, exemples, 7 exercices, évaluation |
| 5 | 📒 Les dictionnaires | cours, exemples, 7 exercices, évaluation |
| 6 | 🎮 **Créer un jeu** | comment on fabrique un jeu, ton jeu *Mission Mars* étape par étape, puis **ton propre jeu** |

## 📖 Mode d'emploi (comme dans le carnet n°1)

1. **Lance la cellule ⚙️ juste en dessous**, une seule fois (« ✅ Moteur prêt ! »).
2. **Exercices** : écris ton code sous la ligne `%%exo` (garde-la !), puis lance la cellule → ✅ **BONNE RÉPONSE** ou ❌ **PAS ENCORE** + explication.
3. **Nouveauté** : dans ce carnet, beaucoup d'exercices te demandent d'écrire une **fonction**. Je l'appelle moi-même avec plein de valeurs différentes pour la tester. Tu n'as pas besoin de l'appeler toi-même dans la cellule `%%exo`.
4. `indice("4.1")` pour un indice, `solution("4.1")` après un essai, `bilan(4)` pour ta note, `progression()` et `export()` comme avant.
5. `%%ralenti` marche toujours : mets-le au-dessus de n'importe quel code pour le voir s'exécuter étape par étape.

⚠️ **iPhone** : pense à désactiver « Ponctuation intelligente » et « Majuscules auto » dans **Réglages → Général → Clavier**.

In [ ]:
# ⚙️ MOTEUR DES EXERCICES — lance cette cellule EN PREMIER (tu n'as pas besoin de la lire)
# Si tu fermes l'application, relance-la avant de continuer.
import builtins, sys, json, base64, re, unicodedata, random

_DATA = json.loads(base64.b64decode("...").decode("utf-8"))
_FICHIER = "progression_python.json"
try:
    with open(_FICHIER, encoding="utf-8") as _f:
        _ETAT = json.load(_f)
except Exception:
    _ETAT = {}
for _k in ("essais", "reussi", "indices", "code", "solutions", "qcm", "bilans", "ok_code"):
    _ETAT.setdefault(_k, {})


def _sauver():
    try:
        with open(_FICHIER, "w", encoding="utf-8") as f:
            json.dump(_ETAT, f, ensure_ascii=False, indent=1)
    except Exception:
        pass


class _TropDInputs(Exception):
    pass


class _Infini(Exception):
    pass


class _Manque(Exception):
    pass


class _Resultat:
    def __init__(self):
        self.out = ""
        self.transcript = ""
        self.ns = {}
        self.used = 0
        self.erreur = None
        self.inputs = []
        self.code = ""

    @property
    def lignes(self):
        return [l.rstrip() for l in self.out.rstrip().split("\n")] if self.out.strip() else []

    @property
    def pleines(self):
        return [l.strip() for l in self.lignes if l.strip()]


def _plain(t):
    return unicodedata.normalize("NFD", str(t)).encode("ascii", "ignore").decode().lower()


def _nums(t):
    return [float(x) if "." in x else int(x) for x in re.findall(r"-?\d+(?:\.\d+)?", str(t))]


def _dernier(t):
    n = _nums(t)
    return n[-1] if n else None


def _entiers_seuls(r):
    return [int(l) for l in r.pleines if re.fullmatch(r"-?\d+", l)]


def _nettoyer(code):
    for a, b in (("“", '"'), ("”", '"'), ("„", '"'), ("«", '"'), ("»", '"'),
                 ("‘", "'"), ("’", "'"), (" ", " "), (" ", " "), ("−", "-"),
                 ("–", "-"), ("—", "-"), ("…", "..."), ("\t", "    ")):
        code = code.replace(a, b)
    return code


def _executer(code, inputs, limite=300000, avant=""):
    r = _Resultat()
    r.inputs = list(inputs)
    r.code = code
    file = list(inputs)

    def faux_input(prompt=""):
        if not file:
            raise _TropDInputs()
        v = file.pop(0)
        r.used += 1
        r.transcript += str(prompt) + str(v) + "\n"
        return str(v)

    def faux_print(*args, sep=" ", end="\n", file=None, flush=False):
        t = sep.join(str(a) for a in args) + end
        r.out += t
        r.transcript += t

    compteur = [0]

    def traceur(frame, event, arg):
        if frame.f_code.co_filename != "<ton code>":
            return None
        if event == "line":
            compteur[0] += 1
            if compteur[0] > limite:
                raise _Infini()
        return traceur

    try:
        prog = compile(code, "<ton code>", "exec")
    except SyntaxError as e:
        r.erreur = e
        return r
    ns = {"__name__": "__main__", "input": faux_input, "print": faux_print}
    r.ns, r._traceur, r._compteur, r._file = ns, traceur, compteur, file
    random.seed(2026)
    if avant:
        exec(compile(avant, "<étapes précédentes>", "exec"), ns)
    ancienne = sys.gettrace()
    sys.settrace(traceur)
    try:
        exec(prog, ns)
    except (Exception, SystemExit) as e:
        r.erreur = e
    finally:
        sys.settrace(ancienne)
    return r


def _appel(r, nom, *args, inputs=None):
    """Appelle la fonction de l'élève (avec la protection contre les boucles infinies)."""
    if inputs is not None:
        r._file[:] = list(inputs)
        r.inputs = list(inputs)
        r.used = 0
    f = r.ns.get(nom)
    if not callable(f):
        raise _Manque(nom)
    avant = len(r.out)
    r._compteur[0] = 0
    ancienne = sys.gettrace()
    sys.settrace(r._traceur)
    try:
        v = f(*args)
    finally:
        sys.settrace(ancienne)
    return v, r.out[avant:]


def _fmt(args):
    return "(" + ", ".join(repr(a) for a in args) + ")"


def _teste(r, nom, cas, strict_bool=False, approx=False):
    """cas = [(arguments, valeur attendue), ...]"""
    for args, att in cas:
        v, sortie = _appel(r, nom, *[a.copy() if isinstance(a, (list, dict)) else a for a in args])
        if v is None and att is not None:
            if sortie.strip():
                return f"{nom}{_fmt(args)} affiche « {sortie.strip()} » mais ne RENVOIE rien. Utilise return au lieu de print."
            return f"{nom}{_fmt(args)} ne renvoie rien (None). As-tu oublié le return ?"
        if strict_bool and not isinstance(v, bool):
            return f"{nom}{_fmt(args)} renvoie {v!r}. Ta fonction doit renvoyer True ou False."
        ok = abs(v - att) < 1e-6 if approx and isinstance(v, (int, float)) and not isinstance(v, bool) else v == att
        if not ok:
            return f"{nom}{_fmt(args)} devrait renvoyer {att!r}, et ta fonction renvoie {v!r}."
    return None


def _ligne_erreur(e):
    if isinstance(e, SyntaxError):
        return e.lineno
    tb, ligne = e.__traceback__, None
    while tb is not None:
        if tb.tb_frame.f_code.co_filename == "<ton code>":
            ligne = tb.tb_lineno
        tb = tb.tb_next
    return ligne


def _explique(e, inputs):
    ligne = _ligne_erreur(e)
    ou = f" (regarde la ligne {ligne})" if ligne else ""
    nom, msg = type(e).__name__, str(e)
    if isinstance(e, _Infini):
        return "Ton programme tourne sans s'arrêter : c'est sûrement une boucle infinie 🔁. Est-ce que la variable de ta condition change bien à chaque tour ?"
    if isinstance(e, _TropDInputs):
        return ("Ton programme pose plus de questions (input) que prévu. J'ai répondu : "
                + (", ".join(f"« {x} »" for x in inputs) if inputs else "rien, car cet exercice n'a pas besoin d'input()")
                + ". Vérifie la condition de ta boucle, ou le nombre de input().")
    if isinstance(e, (IndentationError, TabError)):
        return "Problème de décalage (indentation)" + ou + " : les lignes dans une boucle ou un if sont décalées de 4 espaces, les autres non."
    if isinstance(e, SyntaxError):
        return ("Erreur de syntaxe" + ou + " : as-tu oublié les deux-points « : » à la fin du while, for ou if ? "
                "Une parenthèse ou un guillemet pas fermé ? Un « = » au lieu de « == » dans une condition ?")
    if isinstance(e, NameError):
        n = getattr(e, "name", None) or msg
        if isinstance(n, str) and n[:1].isupper() and n.lower() in ("while", "for", "if", "print", "input", "int", "else", "elif", "range", "len"):
            return f"Ton clavier a mis une majuscule à « {n} »" + ou + f". En Python, il faut l'écrire en minuscules : « {n.lower()} »."
        return f"Python ne connaît pas « {n} »" + ou + ". Vérifie l'orthographe (majuscules comprises) et que la variable est créée AVANT d'être utilisée."
    if isinstance(e, _Manque):
        return f"Je ne trouve pas de fonction qui s'appelle « {e} ». Vérifie son nom : def {e}(...):"
    if isinstance(e, TypeError) and "argument" in msg:
        return "Ta fonction n'a pas le bon nombre de paramètres" + ou + f" : {msg}"
    if isinstance(e, TypeError) and "str" in msg:
        return "Tu mélanges du texte et des nombres" + ou + ". Rappel : input() donne du texte. Pour un nombre, utilise int(input(...))."
    if isinstance(e, ValueError) and "int()" in msg:
        return "int() n'a pas réussi à transformer ce texte en nombre" + ou + ". As-tu mis int() sur une réponse qui n'est pas un nombre ?"
    if isinstance(e, IndexError):
        return "Tu demandes une case qui n'existe pas" + ou + ". Les indices commencent à 0, et la dernière case est à l'indice len(...) - 1."
    if isinstance(e, ZeroDivisionError):
        return "Division par zéro" + ou + " : c'est impossible, même pour Python 😅"
    if isinstance(e, AttributeError):
        return "Cette fonction n'existe pas pour ce type de donnée" + ou + f" : {msg}"
    return f"Ton programme a planté avec l'erreur {nom}" + ou + f" : {msg}"


def _affiche_transcript(r):
    if r is None:
        return
    lignes = r.transcript.rstrip("\n").split("\n") if r.transcript.strip() else []
    rep = ", ".join(f"« {x} »" for x in r.inputs)
    print("📺 Ce que fait ton programme" + (f" (j'ai répondu {rep} à ta place)" if r.inputs else "") + " :")
    if not lignes:
        print("   (rien ne s'affiche)")
    for l in lignes[:30]:
        print("   │ " + l)
    if len(lignes) > 30:
        print(f"   │ … ({len(lignes) - 30} lignes de plus)")
    print()


def _verifier(ident, code):
    ex = _EXOS.get(ident)
    if ex is None:
        print(f"❓ Je ne connais pas l'exercice « {ident} ». Vérifie le numéro après %%exo.")
        return
    code = _nettoyer(code)
    utiles = [l for l in code.split("\n") if l.strip() and not l.strip().startswith("#")]
    if len(utiles) <= ex.get("depart", 0):
        print("✍️ Écris ton code sous la ligne %%exo, puis relance la cellule ▶")
        return
    _ETAT["essais"][ident] = _ETAT["essais"].get(ident, 0) + 1
    _ETAT["code"][ident] = code
    probleme, premier, faits = None, None, 0
    maj = re.search(r"^\s*(While|For|If|Elif|Else|Print|Input|Int|Range|Len)\b", code, re.M)
    if maj:
        probleme = f"Ton clavier a mis une majuscule à « {maj.group(1)} ». En Python, il faut l'écrire en minuscules : « {maj.group(1).lower()} »."
    for motif, msg in ex.get("oblige", []):
        if not probleme and not re.search(motif, code):
            probleme = msg
    for motif, msg in ex.get("interdit", []):
        if re.search(motif, code):
            probleme = msg
    avant = ""
    for i in ex.get("avant", []):
        if i not in _ETAT["ok_code"]:
            probleme = probleme or f"Réussis d'abord l'étape {i} : j'ai besoin de ta fonction de cette étape."
        else:
            avant += _ETAT["ok_code"][i] + "\n"
    if ex.get("statique"):
        r = _Resultat()
        r.code = code
        try:
            compile(code, "<ton code>", "exec")
        except SyntaxError as e:
            probleme = probleme or _explique(e, [])
        probleme = probleme or ex["check"](r)
        if probleme is None:
            faits = 1
    for t in ([] if ex.get("statique") else ex["tests"]):
        c = code
        if "swap" in t:
            ancien, nouveau = t["swap"]
            if ancien not in c:
                continue
            c = c.replace(ancien, nouveau)
        r = _executer(c, [] if ex.get("appel") else t.get("inputs", []), avant=avant)
        r.test = t
        if premier is None:
            premier = r
        if probleme:
            break
        if r.erreur is not None:
            probleme = _explique(r.erreur, t.get("inputs", [])) + ex.get("astuce", {}).get(type(r.erreur).__name__, "")
            break
        try:
            m = ex["check"](r)
        except (Exception, SystemExit) as e:
            m = _explique(e, t.get("inputs", [])) + ex.get("astuce", {}).get(type(e).__name__, "")
        if m:
            probleme = m
            break
        faits += 1
    _affiche_transcript(premier)
    bloc = ident.split(".")[0].lstrip("E")
    if probleme is None:
        _ETAT["reussi"][ident] = True
        _ETAT["ok_code"][ident] = code
        print("✅ BONNE RÉPONSE ! Bravo 🎉")
        if faits > 1:
            print(f"   J'ai testé ton programme {faits} fois, avec des réponses différentes : tout est bon.")
        if ex["type"] == "defi":
            print("   🔥 Défi relevé ! Tu peux être fier de toi.")
        tous = [k for k, v in _EXOS.items() if k.split(".")[0].lstrip("E") == bloc and v["type"] != "eval"]
        ok = sum(1 for k in tous if _ETAT["reussi"].get(k))
        print(f"   Progression du bloc {bloc} : {ok} / {len(tous)} exercices réussis.")
    else:
        _ETAT["reussi"].setdefault(ident, False)
        print("❌ PAS ENCORE… mais tu progresses !")
        print("   👉 " + probleme)
        if ex["type"] != "eval":
            print(f'   💡 Un indice ? Lance la cellule indice("{ident}") juste en dessous.')
    _sauver()


def indice(ident):
    """Affiche le prochain indice d'un exercice."""
    ex = _EXOS.get(ident)
    if ex is None:
        print(f"❓ Je ne connais pas l'exercice « {ident} ».")
        return
    if ex["type"] == "eval":
        print("🙅 Pas d'indice pendant l'évaluation. Tu peux le faire, j'ai confiance en toi !")
        return
    h = _DATA["indices"][ident]
    k = min(_ETAT["indices"].get(ident, 0) + 1, len(h))
    _ETAT["indices"][ident] = k
    for i in range(k):
        print(f"💡 Indice {i + 1}/{len(h)} : {h[i]}")
    if k < len(h):
        print(f"\n   (Relance cette cellule pour l'indice {k + 1}.)")
    else:
        print(f'\n   C\'était le dernier indice. Toujours bloqué ? solution("{ident}") te montre une réponse.')
    _sauver()


def solution(ident, sur=False):
    """Montre une solution expliquée (après au moins un essai)."""
    ex = _EXOS.get(ident)
    if ex is None:
        print(f"❓ Je ne connais pas l'exercice « {ident} ».")
        return
    bloc = ident.split(".")[0].lstrip("E")
    if ex["type"] == "eval" and not _ETAT["bilans"].get(bloc):
        print(f"🔒 Les solutions de l'évaluation se débloquent après bilan({bloc}).")
        return
    if not _ETAT["essais"].get(ident):
        print("🔒 Essaie d'abord au moins une fois (lance ta cellule %%exo), même si ce n'est pas juste.")
        return
    if not sur:
        print("🤔 Tu es sûr ? Chercher encore un peu, c'est comme ça qu'on progresse le plus.")
        print(f'   Si tu veux vraiment voir la réponse, lance : solution("{ident}", sur=True)')
        return
    s = _DATA["solutions"][ident]
    print("📖 Une solution possible (il y en a souvent d'autres !) :\n")
    for l in s["code"].split("\n"):
        print("    " + l)
    print("\n🧠 Explications :")
    for i, x in enumerate(s["explications"], 1):
        print(f"   {i}. {x}")
    print("\n✍️ Maintenant, efface ton code et réécris la solution SANS la regarder, puis relance ta cellule.")
    _ETAT["solutions"][ident] = True
    _sauver()


def qcm(ident, reponse):
    """Répond à une question de QCM, par exemple qcm("1.Q1", "b")."""
    q = _DATA["qcm"].get(ident)
    if q is None:
        print(f"❓ Je ne connais pas la question « {ident} ».")
        return
    rep = str(reponse).strip().lower()
    if rep not in ("a", "b", "c", "d"):
        print('✍️ Remplace le ? par ta réponse : "a", "b" ou "c", puis relance la cellule.')
        return
    premiere = ident not in _ETAT["qcm"]
    if premiere:
        _ETAT["qcm"][ident] = rep
    if rep == q["bonne"]:
        print("✅ Bonne réponse !" + ("" if premiere else " (mais c'est ta première réponse qui compte pour la note)"))
    else:
        print("❌ Ce n'est pas ça." + ("" if premiere else " (c'est ta première réponse qui compte pour la note)"))
        print(f"   La bonne réponse était « {q['bonne']} ».")
    print("   🧠 " + q["explication"])
    _sauver()


def bilan(bloc):
    """Calcule ta note sur 20 pour l'évaluation d'un bloc."""
    bloc = str(bloc)
    qs = [k for k in _DATA["qcm"] if k.split(".")[0] == bloc]
    ev = [k for k, v in _EXOS.items() if v["type"] == "eval" and k.split(".")[0].lstrip("E") == bloc]
    if not qs:
        print(f"❓ Je ne connais pas le bloc {bloc}.")
        return
    manque = [k for k in qs if k not in _ETAT["qcm"]] + [k for k in ev if not _ETAT["essais"].get(k)]
    if manque:
        print(f"✋ Pas encore ! Termine d'abord l'évaluation du bloc {bloc}. Il te reste : " + ", ".join(manque))
        return
    pts_q = sum(2 for k in qs if _ETAT["qcm"].get(k) == _DATA["qcm"][k]["bonne"])
    pts_e = sum(5 for k in ev if _ETAT["reussi"].get(k))
    note = round(20 * (pts_q + pts_e) / (2 * len(qs) + 5 * len(ev)))
    print(f"📋 BILAN DU BLOC {bloc}")
    print(f"   QCM : {pts_q} / {2 * len(qs)}")
    if ev:
        print(f"   Exercices : {pts_e} / {5 * len(ev)}")
    print(f"   👉 NOTE : {note} / 20")
    if note >= 18:
        print("   🏆 Excellent ! Tu maîtrises ce bloc. Direction le suivant !")
    elif note >= 14:
        print("   🚀 Très bien ! Revois juste ce que tu as raté avant de continuer.")
    elif note >= 10:
        print("   👍 C'est un bon début. Refais les exercices ratés du bloc avant de passer à la suite.")
    else:
        print("   💪 Ce bloc n'est pas encore acquis, et c'est normal. Relis le cours et refais les exercices : tu vas y arriver.")
    _ETAT["bilans"][bloc] = note
    if ev:
        print(f'   🔓 Les solutions de l\'évaluation sont débloquées : solution("E{bloc}.1", sur=True)')
    _sauver()


def progression():
    """Affiche tous tes exercices réussis."""
    for bloc in sorted({k.split(".")[0].lstrip("E") for k in _EXOS}, key=int):
        ids = [k for k in _EXOS if k.split(".")[0].lstrip("E") == bloc]
        print(f"Bloc {bloc} :")
        for k in ids:
            etat = "✅" if _ETAT["reussi"].get(k) else ("❌" if _ETAT["essais"].get(k) else "⬜")
            extra = " 📖" if _ETAT["solutions"].get(k) else ""
            print(f"   {etat} {k:<6} {_EXOS[k]['titre']}{extra}")
        if bloc in _ETAT["bilans"]:
            print(f"   📋 Note de l'évaluation : {_ETAT['bilans'][bloc]} / 20")
    print("\n✅ réussi   ❌ essayé   ⬜ pas encore fait   📖 solution vue")


def export():
    """Affiche tout ton code, à copier et envoyer à Claude."""
    print("# Copie tout ce texte et envoie-le à Claude avec le mot « corrige » 🚀\n")
    for k in _EXOS:
        if k in _ETAT["code"]:
            etat = "✅" if _ETAT["reussi"].get(k) else "❌"
            vu = " 📖 solution vue" if _ETAT["solutions"].get(k) else ""
            print(f"# ===== {k} {_EXOS[k]['titre']} {etat}{vu} =====")
            print(_ETAT["code"][k].strip() + "\n")
    for b, n in _ETAT["bilans"].items():
        print(f"# Évaluation bloc {b} : {n}/20")


# ---------------------------------------------------------------- vérifications (blocs 4, 5, 6)
def _c_4_1(r):
    for p in ("Ada", "Alan"):
        v, s = _appel(r, "dire_bonjour", p)
        if f"bonjour {p.lower()}" not in _plain(s):
            return f"dire_bonjour({p!r}) doit afficher « Bonjour {p} ! », et elle affiche « {s.strip() or 'rien'} »."
    return None


def _c_4_2(r):
    return _teste(r, "carre", [((3,), 9), ((0,), 0), ((-4,), 16), ((12,), 144)])


def _c_4_3(r):
    return _teste(r, "est_pair", [((4,), True), ((7,), False), ((0,), True), ((-3,), False)], strict_bool=True)


def _c_4_4(r):
    return _teste(r, "plus_grand", [((3, 8), 8), ((10, 2), 10), ((5, 5), 5), ((-1, -7), -1)])


def _c_4_5(r):
    return _teste(r, "compte_voyelles", [(("fusee",), 3), (("polytechnique",), 6), (("",), 0), (("xyz",), 1)])


def _c_4_6(r):
    return _teste(r, "moyenne", [(([12, 15, 9],), 12), (([10],), 10), (([1, 2],), 1.5), (([],), 0)], approx=True)


def _c_4_7(r):
    return _teste(r, "est_premier", [((n,), p) for n, p in
                                    [(0, False), (1, False), (2, True), (3, True), (4, False), (9, False),
                                     (17, True), (25, False), (97, True), (100, False), (7919, True)]], strict_bool=True)


def _c_E4_1(r):
    return _teste(r, "aire_rectangle", [((3, 4), 12), ((10, 1), 10), ((0, 5), 0)])


def _c_E4_2(r):
    return _teste(r, "inverse", [(("fusee",), "eesuf"), (("",), ""), (("a",), "a"), (("mit",), "tim")])


def _c_5_1(r):
    f = r.ns.get("fusee")
    w = {"nom": "Ariane", "etages": 3, "carburant": 500}
    if not isinstance(f, dict):
        return "Je ne trouve pas de dictionnaire qui s'appelle fusee."
    if f != w:
        return f"Ton dictionnaire fusee vaut {f}. Il doit valoir {w}."
    if not r.pleines or r.pleines[-1] != "Ariane":
        return "Ton dictionnaire est parfait 👏 Affiche maintenant le nom de la fusée."
    return None


def _c_5_2(r):
    if len(re.findall(r"stock\s*=\s*\{", r.code)) > 1:
        return "Ne réécris pas tout le dictionnaire : modifie-le avec stock[...] = ..."
    s = r.ns.get("stock")
    w = {"eau": 80, "nourriture": 25, "oxygene": 50}
    if s != w:
        return f"À la fin, stock doit valoir {w}, et le tien vaut {s}."
    if "oxygene" not in r.out:
        return "Ton dictionnaire est parfait 👏 Affiche-le avec print(stock)."
    return None


def _c_5_3(r):
    return _teste(r, "traduire", [(("fusee",), "rocket"), (("lune",), "moon"), (("soleil",), "???"), (("terre",), "earth")])


def _c_5_4(r):
    s = r.ns.get("scores")
    if not isinstance(s, dict):
        return "Je ne trouve plus le dictionnaire scores. Ne modifie pas la première ligne."
    for nom, pts in s.items():
        if not any(nom in l and pts in _nums(l) for l in r.pleines):
            return f"Avec {s}, je ne vois pas la ligne « {nom} : {pts} »."
    if len(r.pleines) != len(s):
        return f"Avec {s}, il faut exactement {len(s)} lignes, et ton programme en affiche {len(r.pleines)}."
    return None


def _c_5_5(r):
    return _teste(r, "compter_lettres", [(("fusee",), {"f": 1, "u": 1, "s": 1, "e": 2}), (("aaa",), {"a": 3}), (("",), {})])


def _c_5_6(r):
    return _teste(r, "meilleur", [(({"Ada": 12, "Alan": 18, "Grace": 15},), "Alan"), (({"Solo": 3},), "Solo"),
                                  (({"A": -5, "B": -2, "C": -9},), "B")])


def _c_5_7(r):
    return _teste(r, "anagrammes", [(("chien", "niche"), True), (("mars", "rame"), False), (("listen", "silent"), True),
                                    (("abc", "abcc"), False), (("", ""), True), (("aab", "abb"), False)], strict_bool=True)


def _c_E5_1(r):
    return _teste(r, "inventaire", [((["eau", "eau", "pain"],), {"eau": 2, "pain": 1}), (([],), {}), ((["x"],), {"x": 1})])


def _c_E5_2(r):
    a = {"Ada": "0601", "Alan": "0702"}
    return _teste(r, "chercher", [((a, "Ada"), "0601"), ((a, "Bob"), "Inconnu"), (({}, "X"), "Inconnu")])


def _c_6_1(r):
    vus = set()
    for _ in range(300):
        v, _s = _appel(r, "lancer_de")
        if not isinstance(v, int) or not 1 <= v <= 6:
            return f"lancer_de() a renvoyé {v!r}. Elle doit renvoyer un nombre entier entre 1 et 6."
        vus.add(v)
    if len(vus) < 6:
        return f"En 300 lancers, je n'ai obtenu que {sorted(vus)}. Ton dé doit pouvoir donner 1, 2, 3, 4, 5 et 6 (random.randint(1, 6))."
    return None


def _c_6_2(r):
    return _teste(r, "nouvel_etat", [((), {"carburant": 100, "oxygene": 100, "distance": 0})])


def _c_6_3(r):
    e = {"carburant": 70, "oxygene": 45, "distance": 150}
    v, s = _appel(r, "afficher_etat", e)
    n = _nums(s)
    for k in ("carburant", "oxygene", "distance"):
        if e[k] not in n:
            return f"afficher_etat({e}) doit afficher la valeur de {k} ({e[k]}). Ta fonction affiche « {s.strip() or 'rien'} »."
    return None


def _c_6_4(r):
    e = {"carburant": 100, "oxygene": 100, "distance": 0}
    v, s = _appel(r, "avancer", e)
    w = {"carburant": 92, "oxygene": 95, "distance": 50}
    if e != w:
        if isinstance(v, dict) and v == w:
            return "Ta fonction renvoie un nouveau dictionnaire, mais elle doit MODIFIER directement etat (etat[\"distance\"] = ...)."
        return f"Après avancer, l'état devrait être {w}, et il vaut {e}."
    e = {"carburant": 30, "oxygene": 12, "distance": 400}
    _appel(r, "avancer", e)
    if e != {"carburant": 22, "oxygene": 7, "distance": 450}:
        return "Ta fonction doit enlever 8 au carburant et 5 à l'oxygène, et ajouter 50 à la distance, quel que soit l'état de départ."
    return None


def _c_6_5(r):
    att = {1: {"carburant": 80, "oxygene": 100}, 2: {"carburant": 100, "oxygene": 85}, 6: {"carburant": 115, "oxygene": 100}}
    for de in range(1, 7):
        e = {"carburant": 100, "oxygene": 100, "distance": 200}
        v, s = _appel(r, "evenement", e, de)
        w = att.get(de, {"carburant": 100, "oxygene": 100})
        if e["carburant"] != w["carburant"] or e["oxygene"] != w["oxygene"] or e["distance"] != 200:
            return f"Avec le dé {de}, en partant de 100 carburant et 100 oxygène, je devrais avoir {w['carburant']} carburant et {w['oxygene']} oxygène. J'ai {e['carburant']} et {e['oxygene']}."
        if not isinstance(v, str) or not v.strip():
            return f"Avec le dé {de}, evenement doit RENVOYER un message (une chaîne de caractères), et elle renvoie {v!r}."
    return None


def _c_6_6(r):
    cas = [({"carburant": 50, "oxygene": 50, "distance": 500}, "gagne"), ({"carburant": 0, "oxygene": 50, "distance": 100}, "perdu"),
           ({"carburant": 30, "oxygene": -5, "distance": 450}, "perdu"), ({"carburant": 10, "oxygene": 10, "distance": 450}, ""),
           ({"carburant": 100, "oxygene": 100, "distance": 0}, ""), ({"carburant": 5, "oxygene": 5, "distance": 650}, "gagne")]
    return _teste(r, "partie_finie", [((e,), w) for e, w in cas])


def _c_6_7(r):
    v, s = _appel(r, "jouer", inputs=r.test["inputs"])
    p = _plain(s)
    if "/ 500" not in s and "/500" not in s and not any(str(x) in s for x in (100, 90)):
        return "Je ne vois pas l'état de la fusée pendant la partie : appelle afficher_etat(etat) à chaque tour."
    if "inconnue" in r.test.get("attendu", "") and "inconnue" not in p:
        return "Quand le joueur tape autre chose que a ou r, il faut afficher « Commande inconnue »."
    if not ("bravo" in p or "mars" in p or "echou" in p):
        return "À la fin de la partie, il faut afficher « 🎉 Bravo, tu es arrivé sur Mars ! » ou « 💥 Mission échouée… »."
    if r.used > 30:
        return "Ta partie ne s'arrête pas : la boucle doit tourner tant que partie_finie(etat) renvoie \"\"."
    return None


def _c_6_8(r):
    c = r.code
    crit = [
        (len(re.findall(r"^\s*def\s+\w+\s*\(", c, re.M)) >= 2, "au moins 2 fonctions (def)"),
        (bool(re.search(r"\bwhile\b", c)), "une boucle while (la boucle de jeu)"),
        (bool(re.search(r"\binput\s*\(", c)), "le joueur fait des choix (input)"),
        (bool(re.search(r"\bimport\s+random\b|\bfrom\s+random\b", c)), "du hasard (import random)"),
        (bool(re.search(r"\[|\{", c)), "une liste ou un dictionnaire"),
        (bool(re.search(r"\bif\b", c)), "des règles (if)"),
    ]
    print("📋 Le cahier des charges de ton jeu :")
    for ok, t in crit:
        print(f"   {'✅' if ok else '⬜'} {t}")
    print()
    manque = [t for ok, t in crit if not ok]
    if manque:
        return "Il manque encore : " + ", ".join(manque) + "."
    return None


_EXOS = {
    "4.1": dict(titre="Dire bonjour 👋", type="debut", tests=[{}], check=_c_4_1),
    "4.2": dict(titre="Le carré", type="debut", tests=[{}], check=_c_4_2,
                oblige=[(r"\breturn\b", "Ta fonction doit RENVOYER le résultat avec return (pas juste l'afficher).")]),
    "4.3": dict(titre="Pair ou impair ?", type="debut", tests=[{}], check=_c_4_3),
    "4.4": dict(titre="Le plus grand", type="debut", tests=[{}], check=_c_4_4,
                interdit=[(r"\bmax\s*\(", "Pas le droit à max() ici : utilise un if 😉")]),
    "4.5": dict(titre="Compter les voyelles 🔤", type="fin", tests=[{}], check=_c_4_5),
    "4.6": dict(titre="La moyenne 📊", type="fin", tests=[{}], check=_c_4_6),
    "4.7": dict(titre="🔥 Les nombres premiers", type="defi", tests=[{}], check=_c_4_7),
    "E4.1": dict(titre="L'aire du rectangle", type="eval", tests=[{}], check=_c_E4_1),
    "E4.2": dict(titre="Le mot à l'envers", type="eval", tests=[{}], check=_c_E4_2),
    "5.1": dict(titre="La fiche de la fusée 🚀", type="debut", tests=[{}], check=_c_5_1,
                oblige=[(r"fusee\s*\[\s*[\"']nom[\"']\s*\]", "Affiche le nom avec fusee[\"nom\"].")]),
    "5.2": dict(titre="Le stock de la station 📦", type="debut", tests=[{}], check=_c_5_2, depart=1),
    "5.3": dict(titre="Le traducteur 🇬🇧", type="debut", tests=[{}], check=_c_5_3, depart=1),
    "5.4": dict(titre="Le tableau des scores 🏆", type="debut", check=_c_5_4, depart=1,
                oblige=[(r"\bfor\b", "Utilise une boucle for pour parcourir le dictionnaire.")],
                tests=[{}, {"swap": ('{"Ada": 12, "Alan": 18, "Grace": 15}', '{"Linus": 7, "Margaret": 20}')}]),
    "5.5": dict(titre="Compter les lettres", type="fin", tests=[{}], check=_c_5_5),
    "5.6": dict(titre="Le meilleur joueur 🥇", type="fin", tests=[{}], check=_c_5_6,
                interdit=[(r"\bmax\s*\(", "Pas le droit à max() ici : utilise une boucle 😉"), (r"sort", "Pas le droit de trier ici : utilise une boucle 😉")]),
    "5.7": dict(titre="🔥 Les anagrammes", type="defi", tests=[{}], check=_c_5_7,
                interdit=[(r"sort", "Pour ce défi, pas le droit de trier : utilise un dictionnaire pour compter les lettres 😉")]),
    "E5.1": dict(titre="L'inventaire", type="eval", tests=[{}], check=_c_E5_1),
    "E5.2": dict(titre="L'annuaire 📞", type="eval", tests=[{}], check=_c_E5_2),
    "6.1": dict(titre="Le dé 🎲", type="debut", tests=[{}], check=_c_6_1),
    "6.2": dict(titre="L'état de départ", type="debut", tests=[{}], check=_c_6_2),
    "6.3": dict(titre="Le tableau de bord", type="debut", tests=[{}], check=_c_6_3),
    "6.4": dict(titre="Avancer 🚀", type="debut", tests=[{}], check=_c_6_4),
    "6.5": dict(titre="Les événements ☄️", type="fin", tests=[{}], check=_c_6_5),
    "6.6": dict(titre="Gagné ou perdu ?", type="fin", tests=[{}], check=_c_6_6),
    "6.7": dict(titre="La boucle de jeu 🎮", type="fin", check=_c_6_7, appel=True,
                avant=["6.1", "6.2", "6.3", "6.4", "6.5", "6.6"],
                interdit=[(r"^jouer\s*\(", "Ne lance pas jouer() dans cette cellule : je le fais moi-même pour tester. Pour jouer, utilise la cellule 🎮 en dessous.")],
                tests=[{"inputs": ["a"] * 40}, {"inputs": ["z"] + ["r", "a", "a"] * 15, "attendu": "inconnue"}]),
    "6.8": dict(titre="🔥 Ton propre jeu", type="defi", tests=[], check=_c_6_8, statique=True),
}


def _rejouer(ids, appel=None):
    manque = [i for i in ids if i not in _ETAT["ok_code"] and i not in _ETAT["code"]]
    if manque:
        print("✋ Il te manque encore : " + ", ".join(manque))
        return
    ns = {"__name__": "__main__"}
    try:
        for i in ids:
            exec(compile(_ETAT["ok_code"].get(i) or _ETAT["code"][i], f"<étape {i}>", "exec"), ns)
        if appel:
            ns[appel]()
    except KeyboardInterrupt:
        print("\n⏹️ Partie arrêtée.")
    except Exception as e:
        print("❌ " + _explique(e, []))


def jouer_mission():
    """Lance ton jeu Mission Mars (étapes 6.1 à 6.7)."""
    _rejouer(["6.1", "6.2", "6.3", "6.4", "6.5", "6.6", "6.7"], "jouer")


def jouer_mon_jeu():
    """Lance ton propre jeu (exercice 6.8)."""
    _rejouer(["6.8"])


# ---------------------------------------------------------------- magies Jupyter
def _ralenti(line, cell):
    code = _nettoyer(cell)
    src = code.split("\n")
    maxi = int(line) if line.strip().isdigit() else 80
    try:
        prog = compile(code, "<ton code>", "exec")
    except SyntaxError as e:
        print("❌ " + _explique(e, []))
        return
    etapes = [0]

    def montre(*a, sep=" ", end="\n", **k):
        builtins.print("      📺 affiche :", sep.join(str(x) for x in a))

    ns = {"__name__": "__main__", "print": montre}

    def tr(frame, event, arg):
        if frame.f_code.co_filename != "<ton code>":
            return None
        if event == "line":
            etapes[0] += 1
            if etapes[0] > maxi:
                raise _Infini()
            vs = {k: v for k, v in frame.f_locals.items()
                  if not k.startswith("__") and k not in ("print",) and not callable(v) and type(v).__name__ != "module"}
            vt = ", ".join(f"{k} = {v!r}" for k, v in vs.items()) or "aucune variable"
            builtins.print(f"👉 ligne {frame.f_lineno:>2} │ {src[frame.f_lineno - 1].strip():<32} │ {vt}")
        return tr

    print("🐢 Ton code au ralenti (les variables sont montrées AVANT que la ligne s'exécute) :\n")
    ancienne = sys.gettrace()
    sys.settrace(tr)
    try:
        exec(prog, ns)
        print("\n🏁 Fin du programme.")
    except _Infini:
        print(f"\n⏹️ J'arrête après {maxi} étapes (boucle trop longue ou infinie).")
    except Exception as e:
        print("\n❌ " + _explique(e, []))
    finally:
        sys.settrace(ancienne)


try:
    from IPython.core.magic import register_cell_magic
    register_cell_magic("exo")(lambda line, cell: _verifier(line.strip(), cell))
    register_cell_magic("ralenti")(_ralenti)
    print("✅ Moteur prêt ! Tu peux commencer. 🚀")
    if any(_ETAT["essais"].values()):
        print("   (J'ai retrouvé ta progression précédente : lance progression() pour la voir.)")
except Exception as _e:
    print("⚠️ Je n'arrive pas à créer les cellules %%exo dans cette application :", _e)

---
# 🧩 BLOC 4 — Les fonctions

## 1. À quoi ça sert ?

Tu connais déjà des fonctions : `print()`, `input()`, `len()`… Ce sont des **morceaux de code avec un nom**, qu'on peut utiliser quand on veut.

Maintenant, tu vas **créer les tiennes** ! Ça sert à :
- **ne pas répéter** le même code plusieurs fois ;
- **découper** un gros programme (un jeu !) en petits morceaux simples ;
- **tester** chaque morceau séparément.

## 2. Créer une fonction : `def`

```python
def saluer():
    print("Bonjour, équipage !")
    print("Prêts pour le décollage ?")
```

- `def` veut dire « définir » ;
- puis le **nom** de la fonction, des **parenthèses** et les **deux-points** `:` ;
- le code de la fonction est **décalé** de 4 espaces.

⚠️ **Définir n'est pas exécuter !** Ce code ne fait rien tout seul. Pour l'exécuter, il faut **appeler** la fonction :

```python
saluer()     # affiche les 2 lignes
saluer()     # et encore une fois !
```

## 3. Les paramètres : donner des informations à la fonction

```python
def saluer(prenom):
    print("Bonjour", prenom, "!")

saluer("Ada")      # Bonjour Ada !
saluer("Alan")     # Bonjour Alan !
```

`prenom` est un **paramètre** : une variable qui reçoit la valeur donnée entre parenthèses quand on appelle la fonction. On peut en mettre plusieurs : `def aire(longueur, largeur):`

## 4. ⭐ `return` : renvoyer un résultat

C'est **LA** chose importante de ce bloc :

```python
def double(n):
    return n * 2

x = double(5)       # x vaut 10
print(double(7))    # affiche 14
```

`return` **renvoie** une valeur à celui qui a appelé la fonction. On peut la ranger dans une variable, l'afficher, faire un calcul avec…

| `print` | `return` |
|---|---|
| **affiche** à l'écran | **renvoie** une valeur au programme |
| le programme ne peut pas réutiliser le résultat | on peut réutiliser le résultat : `x = double(5) + 1` |

💡 `return` **arrête** la fonction tout de suite. Une fonction sans `return` renvoie `None` (« rien »).

## 5. Des fonctions qui répondent vrai ou faux

```python
def est_majeur(age):
    if age >= 18:
        return True
    else:
        return False
```

Version courte, qui fait pareil : `return age >= 18`

## 6. 🐢 Une fonction au ralenti

Regarde bien : Python **saute** dans la fonction quand on l'appelle, puis **revient** avec le résultat.

In [ ]:
%%ralenti
def double(n):
    resultat = n * 2
    return resultat

x = double(5)
y = double(x)
print(y)

## 7. Exemples à lancer ▶

**Exemple A : une fonction avec deux paramètres.**

In [ ]:
def aire_rectangle(longueur, largeur):
    return longueur * largeur

print(aire_rectangle(3, 4))
print(aire_rectangle(10, 2) + 1)

**Exemple B : une fonction qui utilise une boucle.**

In [ ]:
def somme_jusqua(n):
    total = 0
    for i in range(1, n + 1):
        total = total + i
    return total

print(somme_jusqua(4))     # 1 + 2 + 3 + 4
print(somme_jusqua(100))

**Exemple C : une fonction qui en appelle une autre.**

In [ ]:
def carre(n):
    return n * n

def somme_des_carres(a, b):
    return carre(a) + carre(b)

print(somme_des_carres(3, 4))     # 9 + 16

## 8. ⚠️ Les erreurs classiques

- Oublier les parenthèses à l'appel : `saluer` au lieu de `saluer()` → rien ne se passe
- Utiliser `print` au lieu de `return` → la fonction renvoie `None`
- Mettre du code **après** un `return` → il n'est jamais exécuté
- Appeler la fonction avec le mauvais nombre de valeurs → `TypeError`

---
## ✍️ Exercices du bloc 4 — on commence doucement

### 🎯 Exercice 4.1 ⭐ — Dire bonjour 👋

Écris une fonction `dire_bonjour(prenom)` qui **affiche** `Bonjour <prenom> !`.

**Exemple d'utilisation :**
```
dire_bonjour("Ada")    →  affiche : Bonjour Ada !
```

In [ ]:
%%exo 4.1
# ✍️ Écris ton code ici, puis lance la cellule ▶

In [ ]:
indice("4.1")   # 💡 lance cette cellule si tu bloques

### 🎯 Exercice 4.2 ⭐ — Le carré

Écris une fonction `carre(n)` qui **renvoie** (avec `return`) le carré de `n`.

**Exemple d'utilisation :**
```
carre(3)   →  renvoie 9
carre(-4)  →  renvoie 16
```

In [ ]:
%%exo 4.2
# ✍️ Écris ton code ici, puis lance la cellule ▶

In [ ]:
indice("4.2")   # 💡 lance cette cellule si tu bloques

### 🎯 Exercice 4.3 ⭐⭐ — Pair ou impair ?

Écris une fonction `est_pair(n)` qui renvoie `True` si `n` est pair, et `False` sinon.

**Exemple d'utilisation :**
```
est_pair(4)  →  renvoie True
est_pair(7)  →  renvoie False
```

In [ ]:
%%exo 4.3
# ✍️ Écris ton code ici, puis lance la cellule ▶

In [ ]:
indice("4.3")   # 💡 lance cette cellule si tu bloques

### 🎯 Exercice 4.4 ⭐⭐ — Le plus grand

Écris une fonction `plus_grand(a, b)` qui renvoie le plus grand des deux nombres, **sans utiliser `max()`**.

**Exemple d'utilisation :**
```
plus_grand(3, 8)   →  renvoie 8
plus_grand(5, 5)   →  renvoie 5
```

In [ ]:
%%exo 4.4
# ✍️ Écris ton code ici, puis lance la cellule ▶

In [ ]:
indice("4.4")   # 💡 lance cette cellule si tu bloques

## 💪 Exercices de fin de bloc 4

### 🎯 Exercice 4.5 ⭐⭐⭐ — Compter les voyelles 🔤

Écris une fonction `compte_voyelles(mot)` qui **renvoie** le nombre de voyelles (a, e, i, o, u, y) du mot.

**Exemple d'utilisation :**
```
compte_voyelles("fusee")  →  renvoie 3
```

In [ ]:
%%exo 4.5
# ✍️ Écris ton code ici, puis lance la cellule ▶

In [ ]:
indice("4.5")   # 💡 lance cette cellule si tu bloques

### 🎯 Exercice 4.6 ⭐⭐⭐ — La moyenne 📊

Écris une fonction `moyenne(liste)` qui renvoie la moyenne des nombres de la liste. Si la liste est vide, elle renvoie `0`.

**Exemple d'utilisation :**
```
moyenne([12, 15, 9])  →  renvoie 12.0
moyenne([])           →  renvoie 0
```

In [ ]:
%%exo 4.6
# ✍️ Écris ton code ici, puis lance la cellule ▶

In [ ]:
indice("4.6")   # 💡 lance cette cellule si tu bloques

## 🔥 Défi du bloc 4

### 🎯 Exercice 4.7 🔥 — Les nombres premiers

Un nombre est **premier** s'il a exactement deux diviseurs : 1 et lui-même. 2, 3, 5, 7, 11, 13… sont premiers. 1 n'est pas premier, et 9 non plus (9 = 3 × 3).

Écris une fonction `est_premier(n)` qui renvoie `True` si `n` est premier, et `False` sinon. C'est un grand classique des concours d'informatique !

**Exemple d'utilisation :**
```
est_premier(7)   →  renvoie True
est_premier(9)   →  renvoie False
est_premier(1)   →  renvoie False
```

In [ ]:
%%exo 4.7
# ✍️ Écris ton code ici, puis lance la cellule ▶

In [ ]:
indice("4.7")   # 💡 lance cette cellule si tu bloques

---
## 📝 ÉVALUATION DU BLOC 4

Pas d'indice ici ! **Ta première réponse au QCM compte pour la note.**

### Partie 1 : QCM (10 points)

**Q1.** Qu'affiche ce code ?
```python
def f(x):
    return x + 1
print(f(4))
```

- **a)** 4
- **b)** 5
- **c)** x + 1

In [ ]:
qcm("4.Q1", "?")   # ✍️ remplace ? par a, b ou c

**Q2.** Que renvoie une fonction qui n'a pas de `return` ?

- **a)** 0
- **b)** None
- **c)** Une erreur

In [ ]:
qcm("4.Q2", "?")   # ✍️ remplace ? par a, b ou c

**Q3.** On a défini `def saluer(nom):`. Comment l'appeler ?

- **a)** saluer
- **b)** saluer("Ada")
- **c)** def saluer("Ada")

In [ ]:
qcm("4.Q3", "?")   # ✍️ remplace ? par a, b ou c

**Q4.** Que vaut x ?
```python
def double(n):
    return n * 2
x = double(double(3))
```

- **a)** 6
- **b)** 12
- **c)** 9

In [ ]:
qcm("4.Q4", "?")   # ✍️ remplace ? par a, b ou c

**Q5.** C'est quoi, un paramètre ?

- **a)** La valeur renvoyée par la fonction
- **b)** Une variable qui reçoit la valeur donnée quand on appelle la fonction
- **c)** Le nom de la fonction

In [ ]:
qcm("4.Q5", "?")   # ✍️ remplace ? par a, b ou c

### Partie 2 : exercices (10 points)

### 🎯 Exercice E4.1  — L'aire du rectangle

Écris une fonction `aire_rectangle(longueur, largeur)` qui renvoie l'aire du rectangle.

**Exemple d'utilisation :**
```
aire_rectangle(3, 4)  →  renvoie 12
```

In [ ]:
%%exo E4.1
# ✍️ Écris ton code ici, puis lance la cellule ▶

### 🎯 Exercice E4.2  — Le mot à l'envers

Écris une fonction `inverse(mot)` qui renvoie le mot écrit à l'envers.

**Exemple d'utilisation :**
```
inverse("fusee")  →  renvoie "eesuf"
```

In [ ]:
%%exo E4.2
# ✍️ Écris ton code ici, puis lance la cellule ▶

### 📋 Ta note du bloc 4

In [ ]:
bilan(4)

---
# 📒 BLOC 5 — Les dictionnaires

## 1. C'est quoi ?

Une liste range des valeurs **numérotées** (0, 1, 2…). Un **dictionnaire** range des valeurs avec des **étiquettes** (des « clés ») :

```python
fusee = {"nom": "Ariane", "etages": 3, "carburant": 500}
```

| Clé | Valeur |
|---|---|
| `"nom"` | `"Ariane"` |
| `"etages"` | `3` |
| `"carburant"` | `500` |

On l'écrit entre **accolades** `{ }`, avec `clé: valeur` séparés par des virgules. C'est comme un vrai dictionnaire : on cherche un **mot** (la clé) pour trouver sa **définition** (la valeur).

## 2. Lire et modifier

```python
fusee["nom"]              # "Ariane"
fusee["carburant"] = 450  # on modifie la valeur
fusee["pilote"] = "Ada"   # nouvelle clé : on l'ajoute !
```

⚠️ Lire une clé qui n'existe pas → **erreur** `KeyError`. Pour éviter ça :
- `"pilote" in fusee` → `True` ou `False` (la clé existe-t-elle ?)
- `fusee.get("vitesse", 0)` → la valeur si la clé existe, sinon `0`

## 3. Parcourir un dictionnaire

```python
scores = {"Ada": 12, "Alan": 18}

for nom in scores:                 # parcourt les CLÉS
    print(nom, scores[nom])

for nom, points in scores.items(): # les clés ET les valeurs
    print(nom, ":", points)
```

`len(scores)` donne le nombre de clés.

## 4. ⭐ La technique la plus utile : compter avec un dictionnaire

```python
mot = "fusee"
compte = {}
for lettre in mot:
    if lettre in compte:
        compte[lettre] = compte[lettre] + 1   # déjà vue : +1
    else:
        compte[lettre] = 1                    # première fois
print(compte)    # {'f': 1, 'u': 1, 's': 1, 'e': 2}
```

Regarde-la au ralenti 🐢 :

In [ ]:
%%ralenti
compte = {}
for lettre in "abba":
    if lettre in compte:
        compte[lettre] = compte[lettre] + 1
    else:
        compte[lettre] = 1
print(compte)

## 5. Exemples à lancer ▶

**Exemple A : la fiche d'un astronaute.**

In [ ]:
astronaute = {"nom": "Thomas", "pays": "France", "missions": 2}
print(astronaute["nom"], "vient de", astronaute["pays"])
astronaute["missions"] = astronaute["missions"] + 1
astronaute["metier"] = "ingénieur"
print(astronaute)

**Exemple B : parcourir.**

In [ ]:
planetes = {"Mercure": 0, "Terre": 1, "Mars": 2, "Jupiter": 95}
for nom, lunes in planetes.items():
    print(nom, "a", lunes, "lune(s)")

**Exemple C : un traducteur avec `.get()`.**

In [ ]:
dico = {"bonjour": "hello", "fusee": "rocket"}
mot = "fusee"
print(dico.get(mot, "je ne connais pas ce mot"))
print(dico.get("chat", "je ne connais pas ce mot"))

## 6. ⚠️ Les erreurs classiques

- Lire une clé qui n'existe pas → `KeyError` (utilise `in` ou `.get()`)
- Oublier les guillemets : `fusee[nom]` au lieu de `fusee["nom"]`
- Confondre `[ ]` (liste) et `{ }` (dictionnaire)

---
## ✍️ Exercices du bloc 5 — on commence doucement

### 🎯 Exercice 5.1 ⭐ — La fiche de la fusée 🚀

Crée un dictionnaire `fusee` avec les clés `"nom"` (valeur `"Ariane"`), `"etages"` (valeur `3`) et `"carburant"` (valeur `500`). Puis affiche le nom de la fusée en utilisant sa clé.

**Résultat attendu :**
```
Ariane
```

In [ ]:
%%exo 5.1
# ✍️ Écris ton code ici, puis lance la cellule ▶

In [ ]:
indice("5.1")   # 💡 lance cette cellule si tu bloques

### 🎯 Exercice 5.2 ⭐ — Le stock de la station 📦

Le dictionnaire `stock` est déjà écrit (ne le réécris pas). **Modifie-le** : passe l'eau à `80` et ajoute une clé `"oxygene"` qui vaut `50`. Puis affiche `stock`.

**Résultat attendu :**
```
{'eau': 80, 'nourriture': 25, 'oxygene': 50}
```

In [ ]:
%%exo 5.2
stock = {"eau": 40, "nourriture": 25}
# ✍️ Écris ton code ici, puis lance la cellule ▶

In [ ]:
indice("5.2")   # 💡 lance cette cellule si tu bloques

### 🎯 Exercice 5.3 ⭐⭐ — Le traducteur 🇬🇧

Le dictionnaire `dico` est déjà écrit. Écris une fonction `traduire(mot)` qui renvoie la traduction anglaise du mot, ou `"???"` si le mot n'est pas dans le dictionnaire.

**Exemple d'utilisation :**
```
traduire("fusee")   →  renvoie "rocket"
traduire("soleil")  →  renvoie "???"
```

In [ ]:
%%exo 5.3
dico = {"fusee": "rocket", "lune": "moon", "etoile": "star", "terre": "earth"}
# ✍️ Écris ton code ici, puis lance la cellule ▶

In [ ]:
indice("5.3")   # 💡 lance cette cellule si tu bloques

### 🎯 Exercice 5.4 ⭐⭐ — Le tableau des scores 🏆

Le dictionnaire `scores` est déjà écrit (ne le modifie pas, je vais le changer pour tester). Avec une boucle `for`, affiche chaque joueur et son score, un par ligne.

**Résultat attendu :**
```
Ada : 12
Alan : 18
Grace : 15
```

In [ ]:
%%exo 5.4
scores = {"Ada": 12, "Alan": 18, "Grace": 15}
# ✍️ Écris ton code ici, puis lance la cellule ▶

In [ ]:
indice("5.4")   # 💡 lance cette cellule si tu bloques

## 💪 Exercices de fin de bloc 5

### 🎯 Exercice 5.5 ⭐⭐⭐ — Compter les lettres

Écris une fonction `compter_lettres(mot)` qui renvoie un dictionnaire avec le nombre de fois que chaque lettre apparaît.

**Exemple d'utilisation :**
```
compter_lettres("fusee")  →  renvoie {'f': 1, 'u': 1, 's': 1, 'e': 2}
```

In [ ]:
%%exo 5.5
# ✍️ Écris ton code ici, puis lance la cellule ▶

In [ ]:
indice("5.5")   # 💡 lance cette cellule si tu bloques

### 🎯 Exercice 5.6 ⭐⭐⭐ — Le meilleur joueur 🥇

Écris une fonction `meilleur(scores)` qui reçoit un dictionnaire `{nom: score}` et renvoie le **nom** du joueur qui a le plus grand score, sans `max()` et sans trier.

**Exemple d'utilisation :**
```
meilleur({"Ada": 12, "Alan": 18, "Grace": 15})  →  renvoie "Alan"
```

In [ ]:
%%exo 5.6
# ✍️ Écris ton code ici, puis lance la cellule ▶

In [ ]:
indice("5.6")   # 💡 lance cette cellule si tu bloques

## 🔥 Défi du bloc 5

### 🎯 Exercice 5.7 🔥 — Les anagrammes

Deux mots sont des **anagrammes** s'ils ont exactement les mêmes lettres, dans un autre ordre : *chien* et *niche*, *listen* et *silent*.

Écris une fonction `anagrammes(a, b)` qui renvoie `True` si les deux mots sont des anagrammes, et `False` sinon. **Sans trier** : utilise des dictionnaires !

**Exemple d'utilisation :**
```
anagrammes("chien", "niche")  →  renvoie True
anagrammes("mars", "rame")    →  renvoie False
```

In [ ]:
%%exo 5.7
# ✍️ Écris ton code ici, puis lance la cellule ▶

In [ ]:
indice("5.7")   # 💡 lance cette cellule si tu bloques

---
## 📝 ÉVALUATION DU BLOC 5

### Partie 1 : QCM (10 points)

**Q1.** Avec `d = {"a": 1, "b": 2}`, que vaut `d["b"]` ?

- **a)** 1
- **b)** 2
- **c)** "b"

In [ ]:
qcm("5.Q1", "?")   # ✍️ remplace ? par a, b ou c

**Q2.** Comment ajouter la clé `"c"` avec la valeur 3 ?

- **a)** d.append("c", 3)
- **b)** d["c"] = 3
- **c)** d + {"c": 3}

In [ ]:
qcm("5.Q2", "?")   # ✍️ remplace ? par a, b ou c

**Q3.** Que se passe-t-il avec `d["z"]` si la clé `"z"` n'existe pas ?

- **a)** Ça donne None
- **b)** Ça donne 0
- **c)** Une erreur KeyError

In [ ]:
qcm("5.Q3", "?")   # ✍️ remplace ? par a, b ou c

**Q4.** Que vaut `len({"x": 10, "y": 20})` ?

- **a)** 2
- **b)** 4
- **c)** 30

In [ ]:
qcm("5.Q4", "?")   # ✍️ remplace ? par a, b ou c

**Q5.** Dans `for cle in d:`, que prend la variable `cle` ?

- **a)** Les clés
- **b)** Les valeurs
- **c)** Les paires (clé, valeur)

In [ ]:
qcm("5.Q5", "?")   # ✍️ remplace ? par a, b ou c

### Partie 2 : exercices (10 points)

### 🎯 Exercice E5.1  — L'inventaire

Écris une fonction `inventaire(objets)` qui reçoit une **liste** d'objets et renvoie un dictionnaire qui compte chaque objet.

**Exemple d'utilisation :**
```
inventaire(["eau", "eau", "pain"])  →  renvoie {"eau": 2, "pain": 1}
```

In [ ]:
%%exo E5.1
# ✍️ Écris ton code ici, puis lance la cellule ▶

### 🎯 Exercice E5.2  — L'annuaire 📞

Écris une fonction `chercher(annuaire, nom)` qui renvoie le numéro de `nom` dans le dictionnaire `annuaire`, ou `"Inconnu"` s'il n'y est pas.

**Exemple d'utilisation :**
```
chercher({"Ada": "0601"}, "Ada")  →  renvoie "0601"
chercher({"Ada": "0601"}, "Bob")  →  renvoie "Inconnu"
```

In [ ]:
%%exo E5.2
# ✍️ Écris ton code ici, puis lance la cellule ▶

### 📋 Ta note du bloc 5

In [ ]:
bilan(5)

---
# 🎮 BLOC 6 — Créer un jeu

Tu connais maintenant **tout ce qu'il faut** pour créer un vrai jeu : les boucles, les conditions, les listes, les dictionnaires et les fonctions. Voyons comment les assembler !

## 1. Les 5 pièces d'un jeu

Presque tous les jeux, du morpion à Minecraft, sont construits avec les mêmes pièces :

| Pièce | Rôle | En Python |
|---|---|---|
| 🧠 **L'état** | tout ce qu'il faut retenir : vies, score, position… | des variables, ou mieux, un **dictionnaire** |
| 🔁 **La boucle de jeu** | répéter chaque tour jusqu'à la fin | `while` |
| 🎮 **Les actions du joueur** | ce que le joueur décide | `input()` |
| 📏 **Les règles** | ce qui se passe selon l'action | `if`, dans des **fonctions** |
| 🎲 **Le hasard** | pour que chaque partie soit différente | le module `random` |

Un tour de boucle de jeu, c'est toujours : **afficher l'état → demander une action → appliquer les règles → vérifier si c'est fini**.

## 2. Le hasard : `random`

```python
import random                        # au début du programme

random.randint(1, 6)                 # un entier au hasard entre 1 et 6 (6 compris !)
random.choice(["pierre", "feuille", "ciseaux"])   # un élément au hasard
```

## 3. Sortir d'une boucle : `break`

`break` arrête la boucle immédiatement. Très pratique avec `while True` (une boucle qui tourne « pour toujours ») :

```python
while True:
    reponse = input("Continuer ? (o/n) ")
    if reponse == "n":
        break          # on sort de la boucle
print("Au revoir !")
```

## 4. Un jeu complet, à lancer ▶

Voici un petit jeu de **pile ou face** avec toutes les pièces. Lis-le, joue, puis modifie-le (par exemple, gagner en 5 points) !

In [ ]:
import random

def lancer_piece():
    return random.choice(["pile", "face"])

etat = {"score": 0, "tours": 0}          # 🧠 l'état

while etat["score"] < 3:                 # 🔁 la boucle de jeu
    print("Score :", etat["score"], "| Tours :", etat["tours"])
    choix = input("pile ou face ? ")     # 🎮 l'action du joueur
    resultat = lancer_piece()            # 🎲 le hasard
    print("La pièce tombe sur", resultat)
    if choix == resultat:                # 📏 les règles
        etat["score"] = etat["score"] + 1
        print("Gagné ! ✅")
    else:
        print("Perdu ❌")
    etat["tours"] = etat["tours"] + 1

print("🏆 Tu as gagné en", etat["tours"], "tours !")

## 5. 🚀 Projet guidé : *Mission Mars*

Tu vas fabriquer un vrai jeu, **morceau par morceau**. Le principe :

> Ta fusée doit parcourir **500 km** pour atteindre Mars. À chaque tour, tu choisis : **(a)vancer** ou **(r)éparer**. Mais attention au carburant, à l'oxygène… et aux événements de l'espace !

Chaque étape est **une fonction**, que je teste toute seule. À l'étape 6.7, tu assembles tout, et tu pourras **jouer à ton jeu** !

L'état du jeu est un dictionnaire :
```python
{"carburant": 100, "oxygene": 100, "distance": 0}
```

### 🎯 Exercice 6.1 ⭐ — Le dé 🎲

Écris une fonction `lancer_de()` qui renvoie un nombre au hasard entre 1 et 6. N'oublie pas `import random` en haut de la cellule !

**Exemple d'utilisation :**
```
lancer_de()  →  renvoie par exemple 4
```

In [ ]:
%%exo 6.1
# ✍️ Écris ton code ici, puis lance la cellule ▶

In [ ]:
indice("6.1")   # 💡 lance cette cellule si tu bloques

### 🎯 Exercice 6.2 ⭐ — L'état de départ

Écris une fonction `nouvel_etat()` qui renvoie le dictionnaire de départ : `{"carburant": 100, "oxygene": 100, "distance": 0}`.

**Exemple d'utilisation :**
```
nouvel_etat()  →  renvoie {"carburant": 100, "oxygene": 100, "distance": 0}
```

In [ ]:
%%exo 6.2
# ✍️ Écris ton code ici, puis lance la cellule ▶

In [ ]:
indice("6.2")   # 💡 lance cette cellule si tu bloques

### 🎯 Exercice 6.3 ⭐ — Le tableau de bord

Écris une fonction `afficher_etat(etat)` qui **affiche** le carburant, l'oxygène et la distance, sur une ligne.

**Exemple d'utilisation :**
```
afficher_etat({"carburant": 70, "oxygene": 45, "distance": 150})
→ affiche : ⛽ 70 | 🫁 45 | 📏 150 / 500 km
```

In [ ]:
%%exo 6.3
# ✍️ Écris ton code ici, puis lance la cellule ▶

In [ ]:
indice("6.3")   # 💡 lance cette cellule si tu bloques

### 🎯 Exercice 6.4 ⭐⭐ — Avancer 🚀

Écris une fonction `avancer(etat)` qui **modifie** l'état : la distance augmente de 50, le carburant baisse de 8 et l'oxygène baisse de 5.

**Exemple d'utilisation :**
```
etat = {"carburant": 100, "oxygene": 100, "distance": 0}
avancer(etat)
→ etat vaut maintenant {"carburant": 92, "oxygene": 95, "distance": 50}
```

In [ ]:
%%exo 6.4
# ✍️ Écris ton code ici, puis lance la cellule ▶

In [ ]:
indice("6.4")   # 💡 lance cette cellule si tu bloques

### 💪 Les étapes plus difficiles

### 🎯 Exercice 6.5 ⭐⭐⭐ — Les événements ☄️

Écris une fonction `evenement(etat, de)` qui applique un événement selon la valeur du dé, puis **renvoie** un message :

| Dé | Événement | Effet |
|---|---|---|
| 1 | ☀️ Tempête solaire | carburant − 20 |
| 2 | ☄️ Pluie de météorites | oxygène − 15 |
| 6 | 🎁 Station-service spatiale | carburant + 15 |
| 3, 4, 5 | Rien à signaler | rien ne change |

**Exemple d'utilisation :**
```
evenement(etat, 1)  →  carburant − 20, renvoie "☀️ Tempête solaire ! -20 carburant"
evenement(etat, 4)  →  rien ne change, renvoie "Rien à signaler."
```

In [ ]:
%%exo 6.5
# ✍️ Écris ton code ici, puis lance la cellule ▶

In [ ]:
indice("6.5")   # 💡 lance cette cellule si tu bloques

### 🎯 Exercice 6.6 ⭐⭐⭐ — Gagné ou perdu ?

Écris une fonction `partie_finie(etat)` qui renvoie :
- `"perdu"` si le carburant **ou** l'oxygène est à 0 ou moins ;
- sinon `"gagne"` si la distance est d'au moins 500 ;
- sinon `""` (une chaîne vide : la partie continue).

**Exemple d'utilisation :**
```
partie_finie({"carburant": 0, "oxygene": 50, "distance": 100})   →  "perdu"
partie_finie({"carburant": 50, "oxygene": 50, "distance": 500})  →  "gagne"
partie_finie({"carburant": 50, "oxygene": 50, "distance": 200})  →  ""
```

In [ ]:
%%exo 6.6
# ✍️ Écris ton code ici, puis lance la cellule ▶

In [ ]:
indice("6.6")   # 💡 lance cette cellule si tu bloques

### 🎯 Exercice 6.7 ⭐⭐⭐ — La boucle de jeu 🎮

Assemble tout ! Écris une fonction `jouer()` (sans paramètre) qui :
1. crée l'état avec `nouvel_etat()` ;
2. **tant que** `partie_finie(etat)` renvoie `""` :
   - affiche l'état avec `afficher_etat(etat)` ;
   - demande `(a)vancer ou (r)éparer ? ` ;
   - si `a` → `avancer(etat)` ; si `r` → oxygène + 10 et carburant − 5 ; sinon → affiche `Commande inconnue` ;
   - lance le dé et affiche le message de `evenement(etat, lancer_de())` ;
3. à la fin, affiche `🎉 Bravo, tu es arrivé sur Mars !` si c'est gagné, sinon `💥 Mission échouée…`.

⚠️ **Tes fonctions des étapes 6.1 à 6.6 sont déjà disponibles** : pas besoin de les réécrire. Et **n'appelle pas `jouer()`** dans la cellule : je la teste moi-même. Pour jouer, utilise la cellule 🎮 en dessous !

**Exemple d'utilisation :**
```
⛽ 100 | 🫁 100 | 📏 0 / 500 km
(a)vancer ou (r)éparer ? a
Rien à signaler.
⛽ 92 | 🫁 95 | 📏 50 / 500 km
...
🎉 Bravo, tu es arrivé sur Mars !
```

In [ ]:
%%exo 6.7
# ✍️ Écris ton code ici, puis lance la cellule ▶

In [ ]:
indice("6.7")   # 💡 lance cette cellule si tu bloques

### 🎮 Joue à Mission Mars !

Quand l'étape 6.7 est réussie, lance cette cellule et joue ! Tape `a` ou `r` à chaque tour.

In [ ]:
jouer_mission()

💡 **Améliore ton jeu !** Quelques idées (dans le bac à sable à la fin, ou en refaisant les étapes) :
- un événement « 👽 Rencontre extraterrestre » ;
- une action « (b)ooster » : +100 km mais −30 carburant ;
- un score à la fin : le carburant qu'il te reste.

---
## 🔥 DÉFI FINAL : crée TON propre jeu !

C'est le moment de devenir **créateur de jeux**. Invente un jeu, et écris-le **tout seul** dans la cellule ci-dessous.

**Le cahier des charges** (je vérifie tout ça quand tu lances la cellule) :
- ✅ au moins **2 fonctions** (`def`)
- ✅ une **boucle de jeu** (`while`)
- ✅ le joueur fait des **choix** (`input`)
- ✅ du **hasard** (`import random`)
- ✅ une **liste** ou un **dictionnaire**
- ✅ des **règles** (`if`)

**Des idées si tu n'en as pas :**
- ✊ **Pierre-feuille-ciseaux** contre l'ordinateur, le premier à 3 points gagne ;
- 🔤 **Le pendu spatial** : deviner un mot lettre par lettre, avec 7 erreurs maximum ;
- ❓ **Le quiz de l'espace** : des questions dans un dictionnaire, un score à la fin ;
- 🤖 **Combat de robots** : chaque robot a des points de vie, les attaques font des dégâts au hasard ;
- 🏁 **La course de fusées** : deux fusées avancent avec des dés, la première à 100 gagne.

**Méthode de pro** : commence par écrire l'état du jeu (un dictionnaire), puis une fonction par règle, puis la boucle de jeu. Teste chaque morceau dans le bac à sable avant d'assembler.

⚠️ **Termine ton code par l'appel de ta fonction principale** (par exemple `jeu()`) : quand tu lances la cellule `%%exo 6.8`, je vérifie seulement le cahier des charges, sans lancer le jeu. Ensuite, joue avec la cellule `jouer_mon_jeu()` en dessous.

In [ ]:
%%exo 6.8
# ✍️ Écris ton jeu ici, puis lance la cellule ▶

In [ ]:
indice("6.8")   # 💡 lance cette cellule si tu manques d'idées

### 🕹️ Joue à ton jeu !

In [ ]:
jouer_mon_jeu()

---
## 📝 ÉVALUATION DU BLOC 6 (QCM)

**Q1.** `random.randint(1, 6)` peut donner…

- **a)** un nombre de 1 à 5
- **b)** un nombre de 1 à 6
- **c)** un nombre de 0 à 6

In [ ]:
qcm("6.Q1", "?")   # ✍️ remplace ? par a, b ou c

**Q2.** Que fait `break` dans une boucle ?

- **a)** Il arrête tout le programme
- **b)** Il fait sortir de la boucle
- **c)** Il passe au tour suivant

In [ ]:
qcm("6.Q2", "?")   # ✍️ remplace ? par a, b ou c

**Q3.** La « boucle de jeu », c'est…

- **a)** une boucle qui répète : afficher, demander une action, appliquer les règles, jusqu'à la fin
- **b)** une fonction qui dessine le jeu
- **c)** le dé du jeu

In [ ]:
qcm("6.Q3", "?")   # ✍️ remplace ? par a, b ou c

**Q4.** Que donne `random.choice(["pierre", "feuille", "ciseaux"])` ?

- **a)** Toujours "pierre"
- **b)** Un des trois, au hasard
- **c)** Les trois à la fois

In [ ]:
qcm("6.Q4", "?")   # ✍️ remplace ? par a, b ou c

**Q5.** Pourquoi découper un jeu en fonctions ?

- **a)** C'est obligatoire en Python
- **b)** Chaque morceau est plus simple à écrire et à tester
- **c)** Le jeu va plus vite

In [ ]:
qcm("6.Q5", "?")   # ✍️ remplace ? par a, b ou c

### 📋 Ta note du bloc 6

In [ ]:
bilan(6)

---
# 🏁 Fin du carnet n°2 !

Tu sais maintenant **créer des fonctions, utiliser des dictionnaires et fabriquer un jeu** 🎮. C'est énorme !

- `progression()` pour voir tout ce que tu as réussi ;
- `export()` : copie tout le texte et envoie-le à Claude avec **« corrige »**. Envoie-moi surtout **ton jeu** : on le mettra dans ton dossier `projets/` sur GitHub, et on pourra l'améliorer ensemble.

In [ ]:
progression()

In [ ]:
export()

## 🧪 Bac à sable

Fais tous les essais que tu veux ici !

In [ ]:
# Essaie ce que tu veux ici 🚀